# Spur CLI-Owned Skill Assets Design

**Decision date:** 2026-08-17  
**Design epic:** `bd-2zk`  
**Target areas:** `crates/spur-cli`, `crates/spur-core/src/skills`, `xtask`, release packaging

## Status

Approved in conversation for specification authoring. Implementation remains gated on review of this written specification.

## Goal

Make `crates/spur-cli/assets/skills` the sole checked-in source of SPUR's bundled skill corpus while keeping skills as ordinary filesystem assets.

This change establishes package ownership. It does **not** embed skill content into the executable and does **not** consolidate the currently separate release artifacts.

## Current State

The repository stores 32 skill directories (75 files, about 712 KiB) under `assets/skills`. The filesystem-backed `SkillCatalog` lives in `spur-core`; development discovery falls back to the repository-level directory. Local installation and release packaging copy that tree to `share/spur/skills`. Tag releases also publish a platform-independent `spur-skills-<version>.tar.gz`, and the npm installer downloads it separately from the platform binary.

## Chosen Design

### Source layout

The complete bundled corpus moves as one directory tree:

```text
crates/spur-cli/
├── Cargo.toml
├── assets/
│   └── skills/
│       ├── <skill-id>/SKILL.md
│       └── <skill-id>/<supporting files>
└── src/
```

There is no repository-root compatibility symlink or duplicate copy. References that describe the actual checked-in source tree must use `crates/spur-cli/assets/skills`.

### Responsibility boundary

- `spur-cli` owns and packages the bundled corpus.
- `spur-core` continues to own validation, catalog loading, override precedence, projection, and reconciliation.
- `xtask` continues to own local install and tag-release assembly.
- Runtime consumers continue to read a directory; no Rust API exposes compiled skill strings.

This avoids a dependency from `spur-core` back to `spur-cli`. The core resolver only knows candidate filesystem paths. Its compile-time development fallback may locate the sibling CLI crate through the workspace layout, but production discovery remains package-relative.

### Installed layout

The stable installed contract remains:

```text
<prefix>/share/spur/skills/<skill-id>/SKILL.md
```

Local installation, cargo-dist, release packaging, and npm extraction keep producing this layout. The move changes the build-time source path, not the installed path.

## Resolution and Compatibility

The effective bundled-root precedence does not change:

1. `SPUR_SKILLS_DIR`
2. layered `[skills].bundled_dir`
3. installed package-relative candidates
4. development/workspace fallback

Package-relative discovery continues to accept `share/spur/skills` and the existing package-local compatibility candidates. The development fallback changes from the repository-root corpus to `crates/spur-cli/assets/skills`.

A generic `<repo>/assets/skills` candidate may remain where it is part of the resolver's external/configuration compatibility contract; it must not be recreated in this repository. Tests that build temporary catalogs under `assets/skills` remain valid when they test that generic contract rather than the SPUR source layout.

Missing-root diagnostics must identify the actual candidates checked and describe the workspace source as CLI-owned rather than claiming the corpus is repository-root owned.

## Packaging Changes

- Include `assets/skills/**` in the `spur-cli` Cargo package contract. Prefer an explicit manifest inclusion rule if needed to make `cargo package --list` stable and reviewable.
- Update cargo-dist's workspace inclusion path from the repository-root asset tree to the CLI-owned tree.
- Change `xtask` local-install and distribution packaging sources to `crates/spur-cli/assets/skills`.
- Continue staging release assets under `share/spur/skills`.
- Continue producing the platform-independent `spur-skills-<version>.tar.gz` in this scope.
- Keep npm's two-download installation behavior in this scope.

## Migration

Use a Git move for the corpus so history remains attributable. Update checked-in path references according to meaning:

- Source-layout documentation and commands: change to the CLI-owned path.
- Runtime installed-layout documentation: keep `share/spur/skills`.
- Generic examples and temporary resolver fixtures: keep `assets/skills` when it denotes an arbitrary configured root.
- Historical specifications and plans: do not rewrite past decisions unless a live instruction would become incorrect; add a superseding note where necessary.

No compatibility symlink and no duplicate corpus are permitted.

## Error Handling

No new runtime error class is required. Existing missing-root, invalid-id, unreadable-file, and frontmatter errors remain authoritative.

The migration must ensure:

- missing-root errors include the new CLI-owned development candidate;
- packaging fails early if `crates/spur-cli/assets/skills` is missing;
- incomplete Cargo or release packages fail verification rather than silently installing a partial catalog;
- overrides and installed package candidates retain their current precedence and diagnostics.

## Verification

Focused verification must cover:

1. `scripts/spur-cargo package -p spur-cli --list` contains every bundled skill entry and supporting file.
2. The source corpus contains exactly one directory for each valid skill id and every directory has a parseable `SKILL.md` with a description.
3. `SkillCatalog::discover` finds the CLI-owned corpus during workspace development.
4. Environment, config, and package-relative candidates still outrank workspace fallback.
5. Local `xtask install` copies the CLI-owned corpus to `share/spur/skills`.
6. `xtask dist` still emits the versioned skills archive with `share/spur/skills/<id>/...` paths.
7. cargo-dist configuration includes the moved asset tree.
8. npm installation tests remain green because the installed layout is unchanged.
9. Repository checks contain no live source-path reference that incorrectly points to the removed root corpus.
10. Targeted Rust tests and formatting run through `scripts/spur-cargo`; never invoke bare `cargo`.

## Non-Goals

- Embedding skill markdown or supporting files into the executable.
- Moving catalog/resolver behavior from `spur-core` into `spur-cli`.
- Changing skill selection, projection, adapter rendering, overrides, or frontmatter.
- Combining platform binaries and skills into one release artifact.
- Changing npm's download model.
- Reorganizing individual skill directories or editing their substantive content.

## Risks and Mitigations

### Hidden root-path assumptions

String references exist in core tests, `xtask`, cargo-dist configuration, repository instructions, and a few skill support documents. Each must be classified by meaning rather than mechanically replaced.

**Mitigation:** add a focused stale-reference check and review every remaining `assets/skills` occurrence.

### Cargo package completeness

Moving under the crate makes Cargo-package ownership clearer, but implicit inclusion rules can be weakened by future manifest edits.

**Mitigation:** make inclusion explicit where practical and test the package file list.

### Resolver ownership confusion

A source move could tempt moving runtime behavior into `spur-cli`, creating duplicated catalog logic or preventing non-CLI consumers from loading skills.

**Mitigation:** keep all catalog behavior in `spur-core`; change only candidate paths and asset sourcing.

### Release regression

The tag-release and cargo-dist paths are distinct and currently rely on different configuration.

**Mitigation:** verify both paths independently and preserve the common installed `share/spur/skills` contract.

## Implementation Boundaries

A plan should use ordered, isolated tasks:

1. **Corpus and resolver contract** — move the tree, update the development fallback, adjust resolver tests, and update live repository instructions.
2. **Install and release packaging** — update `xtask`, cargo-dist inclusion, Cargo package checks, and release tests. Depends on task 1.
3. **Reference audit and verification** — classify remaining path references, run focused and broader checks, and document evidence. Depends on tasks 1 and 2.

The source move and resolver edits are coupled and should be one task. Packaging follows because it consumes the new canonical path. Final audit must run after both.

No NS-Mermaid formal cell is required: this design introduces no state machine, eligibility partition, numeric invariant, or proof obligation. Path precedence is unchanged and is covered by executable tests.

## Acceptance Criteria

- `crates/spur-cli/assets/skills` is the only checked-in bundled corpus.
- Repository-root `assets/skills` does not exist and is not recreated by a symlink.
- Bundled skills remain filesystem-backed and are not compiled into the executable.
- `spur-core` remains the single catalog and projection implementation.
- Development discovery resolves the moved corpus.
- Overrides and configured roots preserve precedence.
- Cargo packaging includes the entire corpus.
- Local install, cargo-dist, tag-release, and npm workflows retain a complete `share/spur/skills` installation.
- All relevant tests pass through repository-prescribed commands.
- Live documentation points authors to the CLI-owned source path.